# sre-gym Max tier — chaos engineering demo

> The Max tier is **not provisioned** in this repo (the cluster cost sits at $40-150/day). This notebook walks through the family spec, chaos library, and reference instance so you can see what the apex tier looks like end-to-end.

Three things to inspect:

1. The **22-service docker-compose stack** that backs `ecommerce_vibecoded_saas`.
2. The **11-pattern chaos library** — composable Litmus/Chaos-Mesh-style fault patterns grounded in real 2025-26 production incidents.
3. The **reference instance** — one fully-specified scenario with two simultaneous chaos patterns (Stripe webhook regression + Supabase RLS silent leak) and an expected 110-action trajectory.

In [ ]:
%%bash
pip install -q pyyaml openenv-core>=0.2.1 pydantic>=2.8 fastapi httpx pandas matplotlib
if [ ! -d /content/sre-enginnerllm ]; then
  git clone https://github.com/dakshdoesdev/sre-enginnerllm.git /content/sre-enginnerllm
fi

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '/content/sre-enginnerllm')
REPO = Path('/content/sre-enginnerllm')

from sre_gym import SREGym, Tier
env = SREGym(tier=Tier.MAX)
for k, v in env.describe().items():
    print(f'  {k:<30} {v}')

## 1. The family spec — `ecommerce_vibecoded_saas`

In [ ]:
import yaml
import pandas as pd

specs = env.list_scenarios()
fam = specs[0]
print(f"Family ID: {fam['id']}")
print(f"\nDescription:\n{fam['description']}")

In [ ]:
# 22-service topology, by kind.
topo = pd.DataFrame(fam['topology']['services'])
print(f'Topology: {len(topo)} services')
print(topo.groupby('kind').size().to_string())
print()
print(topo.to_string(index=False))

In [ ]:
# 50+ action space — Basic + Advanced + Max-only.
max_actions = fam.get('allowed_actions', [])
if max_actions and isinstance(max_actions, list) and isinstance(max_actions[0], dict) and 'inherits' in max_actions[0]:
    print('Inherits:', max_actions[0]['inherits'])
    extra = [a for a in max_actions if isinstance(a, str)]
    print(f'\nMax-only actions ({len(extra)}):')
    for a in extra:
        print(f'  {a}')

In [ ]:
# Scenario population — 42 instances composed from chaos patterns.
pop = fam['scenario_population']
print(f"size: {pop['size']}")
for rule in pop['composition_rules']:
    print(f"  {rule['pattern']:<35} count={rule['count']}")
print(f"\ndifficulty distribution: {pop['difficulty_distribution']}")

## 2. The chaos library — 11 composable fault patterns

In [ ]:
with (REPO / 'sre_gym/max/chaos/ecommerce_chaos_library.yaml').open() as f:
    chaos = yaml.safe_load(f)
patterns = chaos['patterns']
print(f'{len(patterns)} chaos patterns:')
for name, body in patterns.items():
    classification = body.get('classification', '')
    cls = f' [{classification}]' if classification else ''
    print(f'\n  --- {name}{cls} ---')
    print(f'    targets         : {body["targets"]}')
    print(f'    inject.type     : {body["inject"]["type"]}')
    print(f'    canonical recovery: {body["canonical_recovery"]}')
    print(f'    grader_focus    : {body["grader_focus"]}')

In [ ]:
# Composability constraints — what can and can't be combined.
safety = chaos['composition_safety']
print('always_safe_pairs:')
for pair in safety['always_safe_pairs']:
    print(f'  + {pair[0]} + {pair[1]}')
print('\nunsafe_pairs:')
for pair in safety['unsafe_pairs']:
    print(f'  x {pair[0]} + {pair[1]}')
print(f'\nmax_simultaneous_patterns: {safety["max_simultaneous_patterns"]}')

## 3. The reference instance — Stripe regression + Supabase RLS leak

Two chaos patterns composed simultaneously. The agent must recognize them as **independent root causes** and fix the security one (RLS leak) before the reliability one (Stripe webhook), so leak duration is minimized.

In [ ]:
ref = fam['reference_instance']
print(f"ID: {ref['id']}")
print(f"\nDescription:\n{ref['description']}\n")
print(f"Chaos patterns applied: {ref['chaos_patterns_applied']}")
print(f"Expected optimal trajectory length: {ref['expected_optimal_trajectory_length']}")
print(f"Expected wall-clock duration: {ref['expected_wall_clock_duration']}")
print(f"Expected optimal score band: {ref['expected_optimal_score_band']}")
print(f"Human SRE baseline band: {ref['human_baseline_score_band']}")

## 4. The reward model — outcome-scored with a learned critic

In [ ]:
rm = fam['reward_model']
print(f"type: {rm['type']}")
print(f"primary_signal: {rm['primary_signal']}")
print('\nrubric_dimensions (auxiliary shaping):')
for dim, meta in rm['rubric_dimensions'].items():
    sign = meta.get('sign', 'plus')
    sign_str = '+' if sign == 'plus' else '-'
    print(f'  {sign_str}{abs(meta["weight"]):.2f}  {dim}')

## 5. Operator notes — what it costs to actually run this

These notes are the difference between an academic spec and an operator-actionable spec.

In [ ]:
notes = fam['operator_notes']
print(f"cost_estimate         : {notes['cost_estimate']}")
print(f"recommended_hardware  : {notes['recommended_hardware']}")
print('\nisolation_requirements:')
for r in notes['isolation_requirements']:
    print(f'  - {r}')
print('\nreset_safety:')
for r in notes['reset_safety']:
    print(f'  - {r}')

## 6. The docker-compose stack

The 22-service docker-compose file in `sre_gym/max/compose/ecommerce.yaml` brings up the topology locally. Stub images aren't published in this repo (publishing them is a $1-2k registry-cost commitment), but the file is the authoritative shape.

In [ ]:
with (REPO / 'sre_gym/max/compose/ecommerce.yaml').open() as f:
    compose = yaml.safe_load(f)
services = compose['services']
print(f'{len(services)} services in compose stack:')
for svc, body in services.items():
    img = body.get('image', '')
    print(f'  {svc:<28} image={img}')

## 7. Where this fits in the tier story

Max is the apex. The world is real, the actions are real, faults are real, reward is computed from the actual recovery state of the actual stack. The expected outcome — informed by the chaos engineering literature surveyed during design — is that a 32B-70B specialist trained against this tier *matches* Claude Opus on triage and *exceeds* it on the long-tail vibe-coded SaaS failure patterns the chaos library specifically targets.

**That gap is the experimental claim of the Max tier.** This notebook is the proof that the spec exists at the level of detail an enterprise SRE platform team can act on.